In [1]:
from transformers import T5Tokenizer, T5EncoderModel, AutoModelForSeq2SeqLM
import torch
import re

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
tokenizer = T5Tokenizer.from_pretrained("Rostlab/ProstT5", do_lower_case=False) # lowercase important becasue this destinguishes between 3Di and AA
embedding_model = T5EncoderModel.from_pretrained("Rostlab/ProstT5").to(DEVICE)
translation_model = AutoModelForSeq2SeqLM.from_pretrained("Rostlab/ProstT5").to(DEVICE)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

In [4]:
# half only supported on gpu
embedding_model.float() if DEVICE == "cpu" else embedding_model.half()
translation_model.float() if DEVICE == "cpu" else translation_model.half()

T5ForConditionalGeneration(
  (shared): Embedding(150, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(150, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=4096, bias=False)
              (k): Linear(in_features=1024, out_features=4096, bias=False)
              (v): Linear(in_features=1024, out_features=4096, bias=False)
              (o): Linear(in_features=4096, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 32)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=1024, out_features=16384, bias=False)
              (wo): Linear(in_features=16384, out_features=1024, bias=False)
              (dro

## Workflow
### 1. 3Di generation for all sequences

In [74]:
sequences = ["MSGGGDVVCTGWLRKSPPEKKLRRYAWKKRWFILRSGRMSGDPDVLEYYKNDHSKKPLRIINLNFCEQVD", "MKAIMVLFYVMTLTIIGSFSMLLQKAKERQ"]

In [75]:
# lengths are required
max_len = max([len(s) for s in sequences])
min_len = min([len(s) for s in sequences])

In [76]:
input_sequences = [" ".join(list(re.sub(r"[UZOB]", "X", s))) for s in sequences]
input_sequences

['M S G G G D V V C T G W L R K S P P E K K L R R Y A W K K R W F I L R S G R M S G D P D V L E Y Y K N D H S K K P L R I I N L N F C E Q V D',
 'M K A I M V L F Y V M T L T I I G S F S M L L Q K A K E R Q']

In [77]:
# add <AA2fold> for translation to 3Di
input_sequences = ["<AA2fold> " + s for s in input_sequences]
input_sequences

['<AA2fold> M S G G G D V V C T G W L R K S P P E K K L R R Y A W K K R W F I L R S G R M S G D P D V L E Y Y K N D H S K K P L R I I N L N F C E Q V D',
 '<AA2fold> M K A I M V L F Y V M T L T I I G S F S M L L Q K A K E R Q']

In [78]:
# tokenize
ids = tokenizer(input_sequences, add_special_tokens=True, padding="longest", return_tensors="pt").to(DEVICE)

In [79]:
# Generation configuration for "folding" (AA-->3Di)
gen_kwargs_aa2fold = {
                  "do_sample": True,
                  "num_beams": 3,
                  "top_p" : 0.95,
                  "temperature" : 1.2,
                  "top_k" : 6,
                  "repetition_penalty" : 1.2,
}

In [80]:
# translation
with torch.no_grad():
    translations = translation_model.generate(
        input_ids=ids["input_ids"],
        attention_mask=ids["attention_mask"],
        max_length=max_len,
        min_length=min_len,
        early_stopping=True,
        num_return_sequences=1,
        **gen_kwargs_aa2fold
    )

In [81]:
decoded_translations = tokenizer.batch_decode(translations, skip_special_tokens=True)
structure_sequences = ["".join(ts.split(" ")) for ts in decoded_translations]
structure_sequences

['ddpdfdfpdkdwdwdfddddppddtdtdiwiwtwgdcvvvvaaiwiftdndppdpdtpdidgcvpppdd',
 'dvvvvvvvvvvvvvvvvvvvvvvvvvvvvd']

In [82]:
combined_sequences = list(zip(sequences, structure_sequences))
combined_sequences

[('MSGGGDVVCTGWLRKSPPEKKLRRYAWKKRWFILRSGRMSGDPDVLEYYKNDHSKKPLRIINLNFCEQVD',
  'ddpdfdfpdkdwdwdfddddppddtdtdiwiwtwgdcvvvvaaiwiftdndppdpdtpdidgcvpppdd'),
 ('MKAIMVLFYVMTLTIIGSFSMLLQKAKERQ', 'dvvvvvvvvvvvvvvvvvvvvvvvvvvvvd')]

### 2. Embedding generation for AA and 3Di seqs

In [83]:
input_aa_seqs = [s for s, _ in combined_sequences]
input_3di_seqs = [s for _, s in combined_sequences]

In [84]:
input_aa_seqs = [" ".join(list(re.sub(r"[UZOB]", "X", s))) for s in input_aa_seqs]
input_3di_seqs = [" ".join(list(s)) for s in input_3di_seqs]

In [85]:
input_aa_seqs = ["<AA2fold> " + s for s in input_aa_seqs]
input_3di_seqs = ["<fold2AA> " + s for s in input_3di_seqs]

In [86]:
# verify input
input_aa_seqs

['<AA2fold> M S G G G D V V C T G W L R K S P P E K K L R R Y A W K K R W F I L R S G R M S G D P D V L E Y Y K N D H S K K P L R I I N L N F C E Q V D',
 '<AA2fold> M K A I M V L F Y V M T L T I I G S F S M L L Q K A K E R Q']

In [87]:
# verify input
input_3di_seqs

['<fold2AA> d d p d f d f p d k d w d w d f d d d d p p d d t d t d i w i w t w g d c v v v v a a i w i f t d n d p p d p d t p d i d g c v p p p d d',
 '<fold2AA> d v v v v v v v v v v v v v v v v v v v v v v v v v v v v d']

In [101]:
aa_ids = tokenizer(input_aa_seqs, add_special_tokens=True, padding="longest", return_tensors="pt").to(DEVICE)
di_ids = tokenizer(input_3di_seqs, add_special_tokens=True, padding="longest", return_tensors="pt").to(DEVICE)

In [102]:
with torch.no_grad():
    aa_embeddings = embedding_model(aa_ids["input_ids"], attention_mask=aa_ids["attention_mask"])
    di_embeddings = embedding_model(di_ids["input_ids"], attention_mask=di_ids["attention_mask"])

### 3. Processing of embeddings

In [103]:
# inspect shapes
print(f"AA emb: {aa_embeddings.last_hidden_state.shape}")
print(f"3Di emb: {di_embeddings.last_hidden_state.shape}")
print(f"AA attn: {aa_ids['attention_mask']}")
print(f"3Di attn: {di_ids['attention_mask']}")

AA emb: torch.Size([2, 72, 1024])
3Di emb: torch.Size([2, 71, 1024])
AA attn: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
       device='cuda:0')
3Di attn: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [104]:
emb_0 = aa_embeddings.last_hidden_state[0, 1:-1]
emb_0.shape

torch.Size([70, 1024])

In [107]:
mask_0 = aa_ids.attention_mask[0]
mask_0.shape

torch.Size([72])

In [109]:
torch.sum(aa_ids.attention_mask[1])

tensor(32, device='cuda:0')

In [110]:
len(sequences[1])

30